In [1]:
# ===== Notebook 3 setup: restore Step 5-6 outputs + rebuild image cache =====
import os, glob, shutil, time, numpy as np, pandas as pd
from PIL import Image
from multiprocessing import Pool
W = "/kaggle/working"
for d in ["feats", "figures", "students"]:
    cands = [p for p in glob.glob(f"/kaggle/input/**/{d}", recursive=True) if os.path.isdir(p) and "step6" in p]
    assert cands, f"'{d}' not found in wildleaf-step6 - is that dataset attached?"
    shutil.copytree(cands[0], f"{W}/{d}", dirs_exist_ok=True); print(d, "<-", cands[0], "|", len(os.listdir(f"{W}/{d}")), "entries")
assert os.path.exists(f"{W}/feats/kd_targets/summary.csv"), "kd_targets missing - was 7.1 saved?"
PROC = {"pd": "/kaggle/input/datasets/mashfii/plantdoc-clean-thesis/proc",
        "pw": "/kaggle/input/datasets/mashfii/plantwild-clean-thesis/proc"}
os.makedirs(f"{W}/cache", exist_ok=True); S = 256
def load256(path):
    im = Image.open(path).convert("RGB"); w, h = im.size; r = S / min(w, h)
    im = im.resize((max(S, round(w*r)), max(S, round(h*r))), Image.BICUBIC); w, h = im.size
    l, t = (w - S)//2, (h - S)//2
    return np.asarray(im.crop((l, t, l+S, t+S)), dtype=np.uint8)
for tag in ["pd", "pw"]:
    df = pd.read_csv(f"{W}/feats/{tag}_manifest.csv", low_memory=False)
    paths = [os.path.join(PROC[tag], str(s)[str(s).find("/proc/")+6:]) for s in df.proc_path]
    arr = np.lib.format.open_memmap(f"{W}/cache/{tag}_256.npy", mode="w+", dtype=np.uint8, shape=(len(paths), S, S, 3))
    t = time.time()
    with Pool(4) as pool:
        for i, a in enumerate(pool.imap(load256, paths, chunksize=64)): arr[i] = a
    arr.flush(); del arr; print(f"{tag}: cached {len(paths)} in {time.time()-t:.0f}s")
print("Setup DONE.")


feats <- /kaggle/input/datasets/mashfii/wildleaf-step6/feats | 14 entries
figures <- /kaggle/input/datasets/mashfii/wildleaf-step6/figures | 51 entries
students <- /kaggle/input/datasets/mashfii/wildleaf-step6/students | 40 entries
pd: cached 2576 in 9s
pw: cached 18542 in 51s
Setup DONE.


In [2]:
# ===== Step 7.2: KD trainer (train_student.py v4) + KD hyperparameter sweep =====
import os, time, subprocess, pandas as pd
SCRIPT = r'''
import argparse, os, time, json, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, timm
from torchvision.ops import roi_align
from sklearn.metrics import f1_score, accuracy_score
p = argparse.ArgumentParser()
p.add_argument("--data", required=True); p.add_argument("--proto", required=True)
p.add_argument("--model", required=True); p.add_argument("--seed", type=int, default=0)
p.add_argument("--epochs", type=int, default=30); p.add_argument("--bs", type=int, default=64)
p.add_argument("--lr", type=float, default=1e-3); p.add_argument("--wd", type=float, default=0.05)
p.add_argument("--kd_alpha", type=float, default=0.0); p.add_argument("--kd_tau", type=float, default=2.0)
p.add_argument("--feat", default="/kaggle/working/feats"); p.add_argument("--cache", default="/kaggle/working/cache")
p.add_argument("--out", default="/kaggle/working/students"); p.add_argument("--tag_extra", default="")
a = p.parse_args()
torch.manual_seed(a.seed); np.random.seed(a.seed); torch.backends.cudnn.benchmark = True
dev = "cuda"; ps = a.proto.replace("_role", "")
df = pd.read_csv(f"{a.feat}/{a.data}_manifest.csv", low_memory=False)
K = int(df.class_id.max()) + 1
X = torch.from_numpy(np.load(f"{a.cache}/{a.data}_256.npy")).to(dev)
Y = torch.tensor(df.class_id.values, device=dev)
split = {r: torch.tensor(np.where(df[a.proto].values == r)[0], device=dev) for r in ["train", "val", "test"]}
if a.kd_alpha > 0:
    Tcal = pd.read_csv(f"{a.feat}/kd_targets/summary.csv").set_index(["dataset", "protocol"]).loc[(a.data, a.proto), "T_val"]
    TL = torch.tensor(np.load(f"{a.feat}/kd_targets/{a.data}_{ps}_logits.npy").astype(np.float32) / Tcal, device=dev)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
S = 256
def to_float(idx): return X[idx].permute(0, 3, 1, 2).float() / 255.
def augment(x):
    B = x.shape[0]
    area = S * S * torch.empty(B, device=dev).uniform_(0.5, 1.0)
    r = torch.exp(torch.empty(B, device=dev).uniform_(np.log(3/4), np.log(4/3)))
    w = torch.sqrt(area * r).clamp(max=S); h = torch.sqrt(area / r).clamp(max=S)
    x0 = torch.rand(B, device=dev) * (S - w); y0 = torch.rand(B, device=dev) * (S - h)
    boxes = torch.stack([torch.arange(B, device=dev).float(), x0, y0, x0 + w, y0 + h], 1)
    x = roi_align(x, boxes, output_size=224, spatial_scale=1.0, sampling_ratio=2, aligned=True)
    flip = torch.rand(B, device=dev) < 0.5; x[flip] = x[flip].flip(-1)
    u = lambda: torch.empty(B, 1, 1, 1, device=dev).uniform_(0.8, 1.2)
    x = x * u(); m = x.mean((1, 2, 3), keepdim=True); x = (x - m) * u() + m
    g = (0.299*x[:, 0:1] + 0.587*x[:, 1:2] + 0.114*x[:, 2:3]); x = (x - g) * u() + g
    return (x.clamp(0, 1) - MEAN) / STD
def prep_eval(x): return (F.interpolate(x, size=224, mode="bicubic", antialias=True, align_corners=False).clamp(0, 1) - MEAN) / STD
model = timm.create_model(a.model, pretrained=True, num_classes=K).to(dev).to(memory_format=torch.channels_last)
n_params = sum(p.numel() for p in model.parameters())
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
ntr = len(split["train"]); spe = ntr // a.bs; steps = a.epochs * spe; warm = spe
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s+1)/warm if s < warm else 0.5*(1+np.cos(np.pi*(s-warm)/max(1, steps-warm))))
crit = nn.CrossEntropyLoss(label_smoothing=0.1); scaler = torch.amp.GradScaler()
@torch.no_grad()
def predict(r):
    model.eval(); L = []
    for i in range(0, len(split[r]), 256):
        idx = split[r][i:i+256]
        with torch.amp.autocast("cuda"): L.append(model(prep_eval(to_float(idx)).contiguous(memory_format=torch.channels_last)).float())
    return torch.cat(L).cpu().numpy(), Y[split[r]].cpu().numpy()
curve, best, t0 = [], (-1, None, -1), time.time()
for ep in range(a.epochs):
    model.train(); tl = tk = 0.0; perm = split["train"][torch.randperm(ntr, device=dev)]
    for s in range(spe):
        idx = perm[s*a.bs:(s+1)*a.bs]
        x = augment(to_float(idx)).contiguous(memory_format=torch.channels_last); y = Y[idx]
        with torch.amp.autocast("cuda"): out = model(x)
        out = out.float(); ce = crit(out, y); loss = ce
        if a.kd_alpha > 0:
            tau = a.kd_tau
            kl = F.kl_div(F.log_softmax(out / tau, 1), F.softmax(TL[idx] / tau, 1), reduction="batchmean") * tau * tau
            loss = (1 - a.kd_alpha) * ce + a.kd_alpha * kl; tk += kl.item()
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tl += ce.item()
    lv, yv = predict("val"); f1v = f1_score(yv, lv.argmax(1), average="macro")
    curve.append(dict(epoch=ep+1, train_ce=tl/spe, train_kl=tk/spe, val_acc=accuracy_score(yv, lv.argmax(1)), val_f1=f1v, time=time.time()-t0))
    if f1v > best[0]: best = (f1v, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, ep+1)
model.load_state_dict(best[1])
lv, yv = predict("val"); lt, yt = predict("test")
kdtag = f"_kd{a.kd_alpha:g}t{a.kd_tau:g}" if a.kd_alpha > 0 else ""
tag = f"{a.data}_{ps}_{a.model.split('.')[0]}_s{a.seed}_lr{a.lr:g}{kdtag}{a.tag_extra}"
res = dict(tag=tag, data=a.data, proto=a.proto, model=a.model, seed=a.seed, lr=a.lr, epochs=a.epochs, pipeline="gpu_aug",
           method="kd" if a.kd_alpha > 0 else "baseline", kd_alpha=a.kd_alpha, kd_tau=a.kd_tau if a.kd_alpha > 0 else None,
           params_M=n_params/1e6, best_epoch=best[2], val_f1=f1_score(yv, lv.argmax(1), average="macro"),
           test_acc=accuracy_score(yt, lt.argmax(1)), test_f1=f1_score(yt, lt.argmax(1), average="macro"),
           minutes=(time.time()-t0)/60, n_train=ntr)
for d in ["logits", "curves", "ckpt"]: os.makedirs(f"{a.out}/{d}", exist_ok=True)
te = split["test"].cpu().numpy()
np.save(f"{a.out}/logits/{tag}_test.npy", lt.astype(np.float16)); np.save(f"{a.out}/logits/{tag}_val.npy", lv.astype(np.float16))
pd.DataFrame({"row": te, "uid": df.uid.values[te]}).to_csv(f"{a.out}/logits/{tag}_test_uids.csv", index=False)
pd.DataFrame(curve).to_csv(f"{a.out}/curves/{tag}.csv", index=False)
torch.save(best[1], f"{a.out}/ckpt/{tag}.pt")
with open(f"{a.out}/results.jsonl", "a") as f: f.write(json.dumps(res) + "\n")
print("RESULT", json.dumps(res))
'''
open("/kaggle/working/train_student.py", "w").write(SCRIPT)

def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:100]
        log = open(f"/kaggle/working/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", "/kaggle/working/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"/kaggle/working/students/log_{tag}.txt").read()[-2000:])

MB3, EB0 = "mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k"
J = lambda d, m, ep, al=0.0, tau=2.0: ["--data", d, "--proto", "p2_role", "--model", m, "--seed", "0", "--lr", "1e-3",
                                       "--epochs", str(ep), "--kd_alpha", str(al), "--kd_tau", str(tau)]
grid = [(0.5, 2), (0.9, 2), (0.5, 4), (0.9, 4)]
jobs = [J("pw", MB3, 20, al, t) for al, t in grid] + \
       [J("pd", MB3, 30, al, t) for al, t in grid] + [J("pd", MB3, 30), J("pd", EB0, 30)]
t0 = time.time()
for i in range(0, len(jobs), 2):
    run_pair(jobs[i:i+2]); print(f"pair {i//2+1}/{len(jobs)//2} done ({(time.time()-t0)/60:.1f} min)")

R = pd.read_json("/kaggle/working/students/results.jsonl", lines=True)
R = R[(R.proto == "p2_role") & (R.get("pipeline") == "gpu_aug")].copy()
R["method"] = R["method"].fillna("baseline"); R["model"] = R.model.str.split(".").str[0]
pd.set_option("display.width", 200)
for d in ["pw", "pd"]:
    print(f"\n=== KD SWEEP: {d.upper()} P2 (select on val_f1) ===")
    print(R[R.data == d][["model", "method", "kd_alpha", "kd_tau", "best_epoch", "val_f1", "test_f1", "minutes"]]
          .round(4).sort_values(["model", "val_f1"], ascending=[True, False]).to_string(index=False))
print("\nStep 7.2 DONE.")

pair 1/5 done (7.2 min)
pair 2/5 done (14.4 min)
pair 3/5 done (17.3 min)
pair 4/5 done (20.3 min)
pair 5/5 done (24.7 min)

=== KD SWEEP: PW P2 (select on val_f1) ===
                 model   method  kd_alpha  kd_tau  best_epoch  val_f1  test_f1  minutes
       efficientnet_b0 baseline       NaN     NaN          18  0.6576   0.6588  10.6475
 mobilenetv3_large_100       kd       0.5     4.0          16  0.6818   0.7141   6.8209
 mobilenetv3_large_100       kd       0.5     2.0          13  0.6804   0.6981   6.8101
 mobilenetv3_large_100       kd       0.9     4.0          20  0.6710   0.7034   6.9650
 mobilenetv3_large_100       kd       0.9     2.0          19  0.6684   0.6951   6.8920
 mobilenetv3_large_100 baseline       NaN     NaN          20  0.6570   0.6611   6.6205
mobilenetv4_conv_small baseline       NaN     NaN          19  0.5805   0.5764   3.2924
              resnet50 baseline       NaN     NaN          13  0.6689   0.6644  13.2572

=== KD SWEEP: PD P2 (select on val_f1) 

In [3]:
# ===== Step 7.3: main KD grid (alpha=0.5, tau=4) + paired comparison vs baselines =====
import os, time, subprocess, numpy as np, pandas as pd, matplotlib.pyplot as plt
W, FIG = "/kaggle/working", "/kaggle/working/figures"
def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:100]
        log = open(f"{W}/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", f"{W}/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"{W}/students/log_{tag}.txt").read()[-2000:])

MB3, EB0, MB4 = "mobilenetv3_large_100.ra_in1k", "efficientnet_b0.ra_in1k", "mobilenetv4_conv_small.e2400_r224_in1k"
def J(d, pr, m, seed=0, kd=True):
    return ["--data", d, "--proto", pr, "--model", m, "--seed", str(seed), "--lr", "1e-3",
            "--epochs", "30" if d == "pd" else "20", "--kd_alpha", "0.5" if kd else "0", "--kd_tau", "4"]
jobs = [J("pd", f"p3_f{f}_role", m) for m in [MB3, EB0, MB4] for f in range(5)]            # 15 PD KD
jobs += [J("pw", "p2_role", EB0), J("pw", "p2_role", MB4)]                                  # PW KD seed0
jobs += [J("pw", "p2_role", m, s, kd) for m in [MB3, EB0] for s in [1, 2] for kd in [False, True]]  # seeds
jobs.append(None)  # pad to even count (last PW job runs alone)
t0 = time.time()
for i in range(0, len(jobs), 2):
    run_pair([j for j in jobs[i:i+2] if j]); print(f"pair {i//2+1}/{len(jobs)//2} done ({(time.time()-t0)/60:.1f} min)")

# ---------- paired comparison ----------
R = pd.read_json(f"{W}/students/results.jsonl", lines=True)
R = R[R.get("pipeline") == "gpu_aug"].copy()
R["method"] = R["method"].fillna("baseline"); R["model"] = R.model.str.split(".").str[0]
R = R[(R.method == "baseline") | ((R.kd_alpha == 0.5) & (R.kd_tau == 4))]
R = R[((R.data == "pd") & R.proto.str.startswith("p3_")) | ((R.data == "pw") & (R.proto == "p2_role"))]
R["unit"] = np.where(R.data == "pd", R.proto, "seed" + R.seed.astype(str))
R = R.sort_values("minutes").drop_duplicates(["data", "model", "method", "unit"], keep="last")   # one run per cell
P = R.pivot_table(index=["data", "model", "unit"], columns="method", values="test_f1").dropna()
P["delta_pp"] = 100 * (P.kd - P.baseline)
S = P.groupby(["data", "model"]).agg(n=("delta_pp", "size"), baseline=("baseline", "mean"), base_sd=("baseline", "std"),
                                     kd=("kd", "mean"), kd_sd=("kd", "std"), delta_pp=("delta_pp", "mean"),
                                     delta_sd=("delta_pp", "std"), wins=("delta_pp", lambda d: f"{(d > 0).sum()}/{len(d)}")).round(4)
pd.set_option("display.width", 200)
print("\n=== PAIRED UNITS (PD: folds, PW: seeds) ===\n", P.round(4).to_string())
print("\n=== MAIN RESULT: baseline vs KD (test macro-F1) ===\n", S.to_string())
S.to_csv(f"{FIG}/table_kd_main.csv"); P.to_csv(f"{FIG}/table_kd_paired_units.csv")

# ---------- fig7_1 ----------
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG).to_dict("records") if os.path.exists(LOG) else []
NICE = {"mobilenetv4_conv_small": "MobileNetV4-S", "mobilenetv3_large_100": "MobileNetV3-L", "efficientnet_b0": "EfficientNet-B0"}
tp = pd.read_csv(f"{FIG}/table_teacher_probes.csv")
teach = {"pd": tp[(tp.dataset == "pd") & (tp.proto == "p3_role") & (tp.feat == "dino+clip")]["mean"].iloc[0],
         "pw": tp[(tp.dataset == "pw") & (tp.proto == "p2_role") & (tp.feat == "dino+clip")]["mean"].iloc[0]}
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
for ax, d, title in zip(axs, ["pd", "pw"], ["PlantDoc P3 (5 folds)", "PlantWild P2 (seeds)"]):
    s = S.loc[d].reindex([m for m in NICE if m in S.loc[d].index]); x = np.arange(len(s))
    ax.bar(x - .2, s.baseline, .4, yerr=s.base_sd.fillna(0), capsize=3, label="Baseline", color="#8C8C8C")
    ax.bar(x + .2, s.kd, .4, yerr=s.kd_sd.fillna(0), capsize=3, label="KD (DINOv2+CLIP teacher)", color="#4C72B0")
    for i, (dl, n, w) in enumerate(zip(s.delta_pp, s.n, s.wins)): ax.text(i + .2, s.kd.iloc[i] + .025, f"{dl:+.1f} pp\n({w})", ha="center", fontsize=7)
    ax.axhline(teach[d], color="#C44E52", ls="--", label=f"Teacher {teach[d]:.3f}")
    ax.set_xticks(x); ax.set_xticklabels([NICE[m] for m in s.index], rotation=15); ax.set_title(title); ax.set_ylim(0.4, 0.85)
axs[0].set_ylabel("Test macro-F1"); axs[0].legend(fontsize=7, loc="lower left")
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig7_1_kd_main.{e}", dpi=300, bbox_inches="tight")
plt.close(fig); log = [r for r in log if r["file"] != "fig7_1_kd_main"] + [dict(file="fig7_1_kd_main", step="7",
      caption="Baseline vs knowledge distillation (α=0.5, τ=4) from the frozen DINOv2+CLIP teacher; labels show mean paired gain and wins.")]
pd.DataFrame(log).to_csv(LOG, index=False); print("saved fig7_1_kd_main\nStep 7.3 DONE.")

pair 1/13 done (2.8 min)
pair 2/13 done (5.6 min)
pair 3/13 done (9.6 min)
pair 4/13 done (13.7 min)
pair 5/13 done (17.7 min)
pair 6/13 done (19.3 min)
pair 7/13 done (20.8 min)
pair 8/13 done (32.3 min)
pair 9/13 done (39.4 min)
pair 10/13 done (46.5 min)
pair 11/13 done (58.0 min)
pair 12/13 done (69.5 min)
pair 13/13 done (81.2 min)

=== PAIRED UNITS (PD: folds, PW: seeds) ===
 method                                  baseline      kd  delta_pp
data model                  unit                                  
pd   efficientnet_b0        p3_f0_role    0.6595  0.6851    2.5588
                            p3_f1_role    0.6264  0.6251   -0.1232
                            p3_f2_role    0.6062  0.6598    5.3623
                            p3_f3_role    0.6505  0.6942    4.3739
                            p3_f4_role    0.5741  0.6317    5.7599
     mobilenetv3_large_100  p3_f0_role    0.6405  0.6809    4.0442
                            p3_f1_role    0.6203  0.6600    3.9648
            

In [11]:
# ===== Step 7.4: DINOv2-PCA foreground masks (for background-counterfactual consistency) =====
import os, time, traceback, numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
try:
    from transformers import AutoModel
    W, FEAT, FIG = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures"
    OUT = f"{FEAT}/fg"; os.makedirs(OUT, exist_ok=True); dev = "cuda:0"
    dino = AutoModel.from_pretrained("facebook/dinov2-base").to(dev).half().eval()
    MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
    ARR = {t: np.load(f"{W}/cache/{t}_256.npy", mmap_mode="r") for t in ["pd", "pw"]}

    @torch.no_grad()
    def patches(arr, idx):                                    # -> (B, 256 patches, 768)
        x = torch.from_numpy(np.ascontiguousarray(arr[idx])).to(dev).permute(0, 3, 1, 2).float() / 255.
        x = (F.interpolate(x, size=224, mode="bicubic", antialias=True).clamp(0, 1) - MEAN) / STD
        return dino(pixel_values=x.half()).last_hidden_state[:, 1:].float()

    # 1) fit PCA on patches of 1000 random images per dataset
    rng = np.random.default_rng(0); samp = []
    for t in ["pd", "pw"]:
        ids = np.sort(rng.choice(len(ARR[t]), 1000, replace=False))
        for i in range(0, 1000, 125): samp.append(patches(ARR[t], ids[i:i+125]).reshape(-1, 768))
    A = torch.cat(samp); mu = A.mean(0)
    _, _, V = torch.pca_lowrank(A - mu, q=3, center=False); V = V[:, :3]          # 768 x 3
    # orient each PC so centre > border
    Z = ((A - mu) @ V).reshape(-1, 16, 16, 3)
    ring = torch.ones(16, 16, dtype=torch.bool, device=dev); ring[3:13, 3:13] = False
    for k in range(3):
        c, b = Z[:, 5:11, 5:11, k].mean(), Z[:, ring, k].mean()
        if c < b: V[:, k] *= -1
        print(f"PC{k+1}: centre-minus-border = {abs(float(c-b)):.3f}")
    del A, Z, samp; torch.cuda.empty_cache()

    # 2) project every image; save 3 x 16 x 16 scores
    stats = []
    for t in ["pd", "pw"]:
        N = len(ARR[t]); S = np.zeros((N, 3, 16, 16), np.float16); t0 = time.time()
        for i in range(0, N, 128):
            idx = np.arange(i, min(N, i+128))
            S[idx] = ((patches(ARR[t], idx) - mu) @ V).reshape(-1, 16, 16, 3).permute(0, 3, 1, 2).cpu().numpy()
        np.save(f"{OUT}/{t}_pca16.npy", S)
        fg = (S[:, 0] > 0).mean((1, 2))
        stats.append(dict(dataset=t, n=N, fg_area_mean=fg.mean(), fg_area_median=np.median(fg),
                          pct_area_lt5=(fg < .05).mean(), pct_area_gt95=(fg > .95).mean(),
                          centre_fg_rate=(S[:, 0, 6:10, 6:10] > 0).mean(), border_fg_rate=(S[:, 0][:, ring.cpu().numpy()] > 0).mean()))
        print(f"{t}: {N} imgs projected in {time.time()-t0:.0f}s")
    ST = pd.DataFrame(stats).round(4); ST.to_csv(f"{OUT}/mask_stats.csv", index=False)
    print("\n=== FOREGROUND MASK STATS (PC1 > 0) ===\n", ST.to_string(index=False))

    # 3) visual checks: PC1 mask overlays + PC1/PC2/PC3 heatmaps
    def up(s): return F.interpolate(torch.tensor(s, dtype=torch.float32)[None, None], size=256, mode="bilinear", align_corners=False)[0, 0].numpy()
    fig, axs = plt.subplots(4, 8, figsize=(16, 8.4))
    for r, t in enumerate(["pd", "pd", "pw", "pw"]):
        S = np.load(f"{OUT}/{t}_pca16.npy", mmap_mode="r"); ids = rng.choice(len(ARR[t]), 4, replace=False)
        for c, i in enumerate(ids):
            im = ARR[t][i]; m = up(S[i, 0]) > 0
            axs[r, 2*c].imshow(im); axs[r, 2*c].axis("off"); axs[r, 2*c].set_title(f"{t} #{i}", fontsize=7)
            ov = im.copy(); ov[~m] = (ov[~m] * 0.25).astype(np.uint8)
            axs[r, 2*c+1].imshow(ov); axs[r, 2*c+1].axis("off"); axs[r, 2*c+1].set_title(f"fg {m.mean():.0%}", fontsize=7)
    fig.suptitle("Foreground masks (PC1 > 0): original | background darkened", fontsize=10)
    for e in ["png", "pdf"]: fig.savefig(f"{FIG}/figD11_fg_masks.{e}", dpi=200, bbox_inches="tight")
    plt.close(fig)
    fig, axs = plt.subplots(4, 4, figsize=(8, 8))
    S = np.load(f"{OUT}/pw_pca16.npy", mmap_mode="r")
    for r, i in enumerate(rng.choice(len(ARR["pw"]), 4, replace=False)):
        axs[r, 0].imshow(ARR["pw"][i]); axs[r, 0].axis("off")
        for k in range(3): axs[r, k+1].imshow(up(S[i, k]), cmap="RdBu_r"); axs[r, k+1].axis("off"); axs[r, k+1].set_title(f"PC{k+1}", fontsize=8)
    for e in ["png", "pdf"]: fig.savefig(f"{FIG}/figD12_pca_components.{e}", dpi=200, bbox_inches="tight")
    plt.close(fig)
    LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG)
    log = log[~log.file.isin(["figD11_fg_masks", "figD12_pca_components"])]
    log = pd.concat([log, pd.DataFrame([
        dict(file="figD11_fg_masks", step="7", caption="Unsupervised leaf masks from the first principal component of DINOv2 patch features (background darkened)."),
        dict(file="figD12_pca_components", step="7", caption="First three principal components of DINOv2 patch features on PlantWild images.")])])
    log.to_csv(LOG, index=False)
    print("saved figD11_fg_masks, figD12_pca_components\nStep 7.4 DONE.")
except Exception:
    traceback.print_exc(); print("Step 7.4 FAILED - Cell B will still run.")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

PC1: centre-minus-border = 7.995
PC2: centre-minus-border = 2.392
PC3: centre-minus-border = 2.439
pd: 2576 imgs projected in 10s
pw: 18542 imgs projected in 73s

=== FOREGROUND MASK STATS (PC1 > 0) ===
 dataset     n  fg_area_mean  fg_area_median  pct_area_lt5  pct_area_gt95  centre_fg_rate  border_fg_rate
     pd  2576        0.5283          0.5586        0.0182         0.0050          0.7349          0.4342
     pw 18542        0.4397          0.4648        0.1306         0.0042          0.5824          0.3743
saved figD11_fg_masks, figD12_pca_components
Step 7.4 DONE.


In [13]:
# ===== Step 7.4b: significance tests + mask failure cases + backup zip =====
import os, numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from scipy import stats
W, FIG, FEAT = "/kaggle/working", "/kaggle/working/figures", "/kaggle/working/feats"

# 1) significance of KD gains (paired over folds / seeds)
P = pd.read_csv(f"{FIG}/table_kd_paired_units.csv")
rows = []
for (d, m), g in P.groupby(["data", "model"]):
    dlt = g.delta_pp.values
    t = stats.ttest_rel(g.kd, g.baseline) if len(g) > 1 else None
    w = stats.wilcoxon(dlt) if len(g) >= 5 else None
    rows.append(dict(data=d, model=m, n=len(g), mean_gain_pp=dlt.mean(),
                     ci95_low=dlt.mean() - stats.t.ppf(.975, len(g)-1) * dlt.std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else np.nan,
                     ci95_high=dlt.mean() + stats.t.ppf(.975, len(g)-1) * dlt.std(ddof=1) / np.sqrt(len(g)) if len(g) > 1 else np.nan,
                     t_p=t.pvalue if t else np.nan, wilcoxon_p=w.pvalue if w else np.nan))
SIG = pd.DataFrame(rows).round(4); SIG.to_csv(f"{FIG}/table_kd_significance.csv", index=False)
print("=== KD SIGNIFICANCE (paired) ===\n", SIG.to_string(index=False))
allp = P.delta_pp.values; print(f"\nALL paired units: n={len(allp)}, mean gain {allp.mean():.2f} pp, "
      f"wins {(allp > 0).sum()}/{len(allp)}, Wilcoxon p={stats.wilcoxon(allp).pvalue:.2e}")

# 2) mask failure cases on PlantWild
ARR = np.load(f"{W}/cache/pw_256.npy", mmap_mode="r"); S = np.load(f"{FEAT}/fg/pw_pca16.npy", mmap_mode="r")
M = pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False)
area = (S[:, 0] > 0).mean((1, 2)); bad = np.where(area < .05)[0]
print(f"\nPW images with fg area < 5%: {len(bad)}")
print("  top classes:", M.class_name.values[bad].tolist() and pd.Series(M.class_name.values[bad]).value_counts().head(8).to_dict())
print("  healthy share among bad:", round(pd.Series(M.class_name.values[bad]).str.fullmatch(r"[a-z ]+ leaf").mean(), 3),
      "| overall:", round(M.class_name.str.fullmatch(r"[a-z ]+ leaf").mean(), 3))
up = lambda s: F.interpolate(torch.tensor(s, dtype=torch.float32)[None, None], size=256, mode="bilinear", align_corners=False)[0, 0].numpy()
rng = np.random.default_rng(1); pick = rng.choice(bad, 8, replace=False)
fig, axs = plt.subplots(4, 8, figsize=(16, 8.4))
for c, i in enumerate(pick):
    axs[0, c].imshow(ARR[i]); axs[0, c].set_title(M.class_name.values[i][:22], fontsize=6.5)
    for k in range(3): axs[k+1, c].imshow(up(S[i, k]), cmap="RdBu_r", vmin=-abs(S[i, k]).max(), vmax=abs(S[i, k]).max())
    for r in range(4): axs[r, c].axis("off")
for k in range(3): axs[k+1, 0].set_ylabel(f"PC{k+1}")
fig.suptitle("PlantWild mask failures (PC1 area < 5%): image | PC1 | PC2 | PC3 (red = positive)", fontsize=10)
fig.savefig(f"{FIG}/figD13_mask_failures.png", dpi=150, bbox_inches="tight"); plt.close(fig)
print("saved figD13_mask_failures.png")

# 3) backup zip (no checkpoints, no cache)
os.system("cd /kaggle/working && rm -f wildleaf_step7.zip && zip -qr wildleaf_step7.zip students figures feats train_student.py -x 'students/ckpt/*'")
print("zip:", round(os.path.getsize(f"{W}/wildleaf_step7.zip")/1e6), "MB")
print("Step 7.4b DONE.")

=== KD SIGNIFICANCE (paired) ===
 data                  model  n  mean_gain_pp  ci95_low  ci95_high    t_p  wilcoxon_p
  pd        efficientnet_b0  5        3.5864    0.5886     6.5841 0.0293      0.1250
  pd  mobilenetv3_large_100  5        3.5977    1.0956     6.0998 0.0162      0.0625
  pd mobilenetv4_conv_small  5       10.4381    7.2378    13.6384 0.0008      0.0625
  pw        efficientnet_b0  3        5.2980    2.5496     8.0463 0.0142         NaN
  pw  mobilenetv3_large_100  3        5.2422    4.6682     5.8162 0.0006         NaN
  pw mobilenetv4_conv_small  1        8.9886       NaN        NaN    NaN         NaN

ALL paired units: n=22, mean gain 5.85 pp, wins 21/22, Wilcoxon p=9.54e-07

PW images with fg area < 5%: 2421
  top classes: {'corn smut': 271, 'rice leaf': 244, 'rice sheath blight': 237, 'garlic leaf': 169, 'garlic leaf blight': 135, 'corn leaf': 132, 'banana panama disease': 117, 'apple rust': 90}
  healthy share among bad: 0.308 | overall: 0.423
saved figD13_mask_

In [14]:
# ===== Step 7.4c: CLS-attention saliency masks (DINOv2 with registers) vs PC1 =====
import os, time, traceback, numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from transformers import AutoModel
W, FEAT, FIG = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures"; dev = "cuda:0"
OUT = f"{FEAT}/fg"; os.makedirs(OUT, exist_ok=True)
net = AutoModel.from_pretrained("facebook/dinov2-with-registers-base", attn_implementation="eager").to(dev).half().eval()
NREG = getattr(net.config, "num_register_tokens", 4); print("register tokens:", NREG)
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1); STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)
ARR = {t: np.load(f"{W}/cache/{t}_256.npy", mmap_mode="r") for t in ["pd", "pw"]}

@torch.no_grad()
def cls_attn(arr, idx):                                     # -> (B, 16, 16) head-mean CLS attention, last layer
    x = torch.from_numpy(np.ascontiguousarray(arr[idx])).to(dev).permute(0, 3, 1, 2).float() / 255.
    x = (F.interpolate(x, size=224, mode="bicubic", antialias=True).clamp(0, 1) - MEAN) / STD
    att = net(pixel_values=x.half(), output_attentions=True).attentions[-1].float()   # B,H,T,T
    a = att[:, :, 0, 1 + NREG:].mean(1)                                                 # CLS -> patches
    return (a / a.sum(1, keepdim=True)).reshape(-1, 16, 16)

def mass_mask(a, keep=0.6):                                 # smallest patch set holding `keep` of attention mass
    v = a.reshape(len(a), -1); s, i = torch.sort(v, 1, descending=True)
    cm = torch.cumsum(s, 1); k = (cm < keep).sum(1, keepdim=True)
    thr = torch.gather(s, 1, k.clamp(max=v.shape[1]-1)); return (v >= thr).reshape(a.shape)

ring = np.ones((16, 16), bool); ring[3:13, 3:13] = False; stats = []
for t in ["pd", "pw"]:
    N = len(ARR[t]); A = np.zeros((N, 16, 16), np.float16); t0 = time.time()
    for i in range(0, N, 128):
        idx = np.arange(i, min(N, i+128)); A[idx] = cls_attn(ARR[t], idx).cpu().numpy()
    np.save(f"{OUT}/{t}_attn16.npy", A)
    Mk = mass_mask(torch.tensor(A, dtype=torch.float32)).numpy()
    area = Mk.mean((1, 2))
    stats.append(dict(dataset=t, fg_area_mean=area.mean(), fg_area_median=np.median(area),
                      pct_area_lt5=(area < .05).mean(), centre_fg_rate=Mk[:, 6:10, 6:10].mean(), border_fg_rate=Mk[:, ring].mean()))
    print(f"{t}: {N} imgs in {time.time()-t0:.0f}s")
print("\n=== CLS-ATTENTION MASKS (60% mass) ===\n", pd.DataFrame(stats).round(4).to_string(index=False))

# visual comparison on the SAME images as figD11 + figD13 failures
up = lambda s: F.interpolate(torch.tensor(np.asarray(s), dtype=torch.float32)[None, None], size=256, mode="bilinear", align_corners=False)[0, 0].numpy()
pcs = {t: np.load(f"{OUT}/{t}_pca16.npy", mmap_mode="r") for t in ["pd", "pw"]}
att = {t: np.load(f"{OUT}/{t}_attn16.npy", mmap_mode="r") for t in ["pd", "pw"]}
Mpw = pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False)
bad = np.where((pcs["pw"][:, 0] > 0).mean((1, 2)) < .05)[0]
show = [("pd", i) for i in [561, 770, 928, 134, 59, 997]] + [("pw", i) for i in [1302, 14598, 10997, 125]] + \
       [("pw", i) for i in np.random.default_rng(1).choice(bad, 6, replace=False)]
fig, axs = plt.subplots(4, len(show)//2 * 2 // 2, figsize=(2.1 * 8, 9)) if False else plt.subplots(8, 8, figsize=(16, 17))
for n, (t, i) in enumerate(show):
    r, c = (n // 8) * 4, n % 8
    im = ARR[t][i]; a = up(att[t][i]); mk = up(mass_mask(torch.tensor(np.asarray(att[t][i]), dtype=torch.float32)[None])[0].float().numpy()) > .5
    pc = up(pcs[t][i, 0]) > 0
    axs[r, c].imshow(im); axs[r, c].set_title(f"{t} #{i}", fontsize=7)
    axs[r+1, c].imshow(a, cmap="inferno"); axs[r+1, c].set_title("CLS attention", fontsize=7)
    o = im.copy(); o[~mk] = (o[~mk] * .25).astype(np.uint8); axs[r+2, c].imshow(o); axs[r+2, c].set_title(f"attn mask {mk.mean():.0%}", fontsize=7)
    o = im.copy(); o[~pc] = (o[~pc] * .25).astype(np.uint8); axs[r+3, c].imshow(o); axs[r+3, c].set_title(f"PC1 mask {pc.mean():.0%}", fontsize=7)
for a_ in axs.ravel(): a_.axis("off")
fig.suptitle("Attention-based vs PCA foreground masks (same images; bottom block = previous PC1 failures)", fontsize=11)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/figD14_attn_vs_pca_masks.{e}", dpi=150, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "figD14_attn_vs_pca_masks"]
pd.concat([log, pd.DataFrame([dict(file="figD14_attn_vs_pca_masks", step="7",
    caption="Unsupervised foreground masks: DINOv2(with registers) CLS-attention (60% mass) vs first PCA component of patch features.")])]).to_csv(LOG, index=False)
print("saved figD14_attn_vs_pca_masks\nStep 7.4c DONE.")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/224 [00:00<?, ?it/s]

register tokens: 4
pd: 2576 imgs in 14s
pw: 18542 imgs in 103s

=== CLS-ATTENTION MASKS (60% mass) ===
 dataset  fg_area_mean  fg_area_median  pct_area_lt5  centre_fg_rate  border_fg_rate
     pd        0.2706          0.2812        0.0000          0.3892          0.2428
     pw        0.2664          0.2734        0.0001          0.3931          0.2380
saved figD14_attn_vs_pca_masks
Step 7.4c DONE.


In [15]:
# ===== Step 7.5: KD ablations on PW P2 (MobileNetV3-L, seeds 0-1) =====
import os, time, subprocess, numpy as np, pandas as pd, torch, torch.nn.functional as F, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold
W, FEAT, FIG = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures"; dev = "cuda:0"

# ---------- 1. extra teacher targets for pw p2 ----------
df = pd.read_csv(f"{FEAT}/pw_manifest.csv", low_memory=False); y = df.class_id.values; K = int(y.max()) + 1
r = df.p2_role.values; tr = np.where(r == "train")[0]; ev = np.where(np.isin(r, ["val", "test"]))[0]; va = np.where(r == "val")[0]
g = df.dup_group.astype(str).where(df.dup_group.notna(), "solo_" + df.uid.astype(str)).values
WD = pd.read_csv(f"{FEAT}/teacher_probe_results.csv").query("dataset=='pw' and protocol=='p2_role'").set_index("feat").wd
l2 = lambda a: a / np.linalg.norm(a, axis=1, keepdims=True)
L = lambda k: np.load(f"{FEAT}/pw_{k}.npy")
FE = {"_dino": (np.hstack([L("dino_cls"), L("dino_mean")]).astype(np.float32), WD["dino_cls+mean"]),
      "_clip": (L("clip").astype(np.float32), WD["clip"]),
      "_insample": (np.hstack([l2(np.hstack([L("dino_cls"), L("dino_mean")])), l2(L("clip"))]).astype(np.float32), WD["dino+clip"])}
def fit_logits(Xtr, ytr, Xev, wd):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    A = torch.tensor((Xtr - mu) / sd, device=dev); yy = torch.tensor(ytr, device=dev)
    Wt = torch.zeros(A.shape[1], K, device=dev, requires_grad=True); b = torch.zeros(K, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([Wt, b], lr=1, max_iter=300, line_search_fn="strong_wolfe")
    def c():
        opt.zero_grad(); l = F.cross_entropy(A @ Wt + b, yy) + wd * (Wt**2).sum(); l.backward(); return l
    opt.step(c); return (torch.tensor((Xev - mu) / sd, device=dev) @ Wt.detach() + b.detach()).cpu().numpy()
def fit_T(z, yy):
    z = torch.tensor(z, device=dev); yy = torch.tensor(yy, device=dev); lt = torch.zeros(1, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([lt], lr=.1, max_iter=100)
    def c():
        opt.zero_grad(); l = F.cross_entropy(z / lt.exp(), yy); l.backward(); return l
    opt.step(c); return float(lt.exp())
Tval = {}
for src, (X, wd) in FE.items():
    Z = np.zeros((len(df), K), np.float32)
    if src == "_insample":                                   # naive: one probe on all train, predicts its own train images
        Z[np.r_[tr, ev]] = fit_logits(X[tr], y[tr], X[np.r_[tr, ev]], wd)
    else:                                                    # cross-fitted like the main teacher
        for a_, b_ in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X[tr], y[tr], g[tr]):
            Z[tr[b_]] = fit_logits(X[tr[a_]], y[tr[a_]], X[tr[b_]], wd)
        Z[ev] = fit_logits(X[tr], y[tr], X[ev], wd)
    Tval[src] = fit_T(Z[va], y[va]); np.save(f"{FEAT}/kd_targets/pw_p2{src}_logits.npy", Z.astype(np.float16))
    P = torch.softmax(torch.tensor(Z[tr]), 1).numpy()
    print(f"{src:10s} T_val={Tval[src]:.3f}  train acc={np.mean(P.argmax(1) == y[tr]):.4f}  "
          f"mean p(given)={P[np.arange(len(tr)), y[tr]].mean():.4f}")

# ---------- 2. patch trainer: --kd_src, --kd_T ----------
s = open(f"{W}/train_student.py").read()
rep = [('p.add_argument("--kd_alpha", type=float, default=0.0); p.add_argument("--kd_tau", type=float, default=2.0)',
        'p.add_argument("--kd_alpha", type=float, default=0.0); p.add_argument("--kd_tau", type=float, default=2.0)\n'
        'p.add_argument("--kd_src", default=""); p.add_argument("--kd_T", type=float, default=-1.0)'),
       ('    Tcal = pd.read_csv(', '    Tcal = a.kd_T if a.kd_T > 0 else pd.read_csv('),
       ('np.load(f"{a.feat}/kd_targets/{a.data}_{ps}_logits.npy")', 'np.load(f"{a.feat}/kd_targets/{a.data}_{ps}{a.kd_src}_logits.npy")'),
       ('kdtag = f"_kd{a.kd_alpha:g}t{a.kd_tau:g}" if a.kd_alpha > 0 else ""',
        'kdtag = (f"_kd{a.kd_alpha:g}t{a.kd_tau:g}{a.kd_src}" + (f"_T{a.kd_T:g}" if a.kd_T > 0 else "")) if a.kd_alpha > 0 else ""'),
       ('method="kd" if a.kd_alpha > 0 else "baseline",', 'method="kd" if a.kd_alpha > 0 else "baseline", kd_src=a.kd_src, kd_T=a.kd_T,')]
if "--kd_src" not in s:
    for old, new in rep:
        assert s.count(old) == 1, f"patch target not found: {old[:60]}"
        s = s.replace(old, new)
    open(f"{W}/train_student.py", "w").write(s); print("trainer patched")

# ---------- 3. ablation runs ----------
def run_pair(jobs):
    procs = []
    for gpu, args in enumerate(jobs):
        tag = "_".join(x for x in args if not x.startswith("--"))[:110]
        log = open(f"{W}/students/log_{tag}.txt", "w")
        procs.append((subprocess.Popen(["python", f"{W}/train_student.py", *args],
                      env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}, stdout=log, stderr=subprocess.STDOUT), tag))
    for p, tag in procs:
        p.wait()
        if p.returncode != 0: print("FAILED", tag, open(f"{W}/students/log_{tag}.txt").read()[-2000:])
MB3 = "mobilenetv3_large_100.ra_in1k"
def J(seed, src="", T=-1.0, al=0.5):
    return ["--data", "pw", "--proto", "p2_role", "--model", MB3, "--seed", str(seed), "--lr", "1e-3", "--epochs", "20",
            "--kd_alpha", str(al), "--kd_tau", "4", "--kd_src", src, "--kd_T", str(T)]
ABL = {"DINOv2-only teacher": dict(src="_dino"), "CLIP-only teacher": dict(src="_clip"),
       "In-sample teacher (no cross-fit)": dict(src="_insample", T=Tval["_insample"]),
       "No calibration (T=1)": dict(T=1.0), "alpha=1.0 (teacher only)": dict(al=1.0)}
ABL["DINOv2-only teacher"]["T"] = Tval["_dino"]; ABL["CLIP-only teacher"]["T"] = Tval["_clip"]
jobs = [J(sd, **cfg) for cfg in ABL.values() for sd in [0, 1]]
t0 = time.time()
for i in range(0, len(jobs), 2):
    run_pair(jobs[i:i+2]); print(f"pair {i//2+1}/{len(jobs)//2} done ({(time.time()-t0)/60:.1f} min)")

# ---------- 4. summary ----------
R = pd.read_json(f"{W}/students/results.jsonl", lines=True)
R = R[(R.data == "pw") & (R.proto == "p2_role") & (R.model == MB3) & (R.seed.isin([0, 1])) & (R.get("pipeline") == "gpu_aug")].copy()
for c, d in [("method", "baseline"), ("kd_src", ""), ("kd_T", -1.0)]: R[c] = R[c].fillna(d) if c in R else d
def label(x):
    if x.method == "baseline": return "Baseline (no KD)"
    if x.kd_alpha == 1.0: return "alpha=1.0 (teacher only)"
    if x.kd_src == "_dino": return "DINOv2-only teacher"
    if x.kd_src == "_clip": return "CLIP-only teacher"
    if x.kd_src == "_insample": return "In-sample teacher (no cross-fit)"
    if x.kd_T == 1.0: return "No calibration (T=1)"
    if x.kd_alpha == 0.5 and x.kd_tau == 4: return "WildLeaf-KD (full)"
    return None
R["variant"] = R.apply(label, axis=1); R = R.dropna(subset=["variant"])
R = R.sort_values("minutes").drop_duplicates(["variant", "seed"], keep="last")
A = R.groupby("variant").agg(n=("test_f1", "size"), val_f1=("val_f1", "mean"), test_f1=("test_f1", "mean"),
                             test_sd=("test_f1", "std"), test_acc=("test_acc", "mean")).round(4)
full = A.loc["WildLeaf-KD (full)", "test_f1"]; A["vs_full_pp"] = (100 * (A.test_f1 - full)).round(2)
order = ["Baseline (no KD)", "DINOv2-only teacher", "CLIP-only teacher", "In-sample teacher (no cross-fit)",
         "No calibration (T=1)", "alpha=1.0 (teacher only)", "WildLeaf-KD (full)"]
A = A.reindex([o for o in order if o in A.index]); pd.set_option("display.width", 200)
print("\n=== KD ABLATIONS: PlantWild P2, MobileNetV3-L, seeds 0-1 ===\n", A.to_string())
A.to_csv(f"{FIG}/table_kd_ablations.csv")

fig, ax = plt.subplots(figsize=(7, 3.4))
cols = ["#8C8C8C"] + ["#DD8452"] * (len(A) - 2) + ["#4C72B0"]
ax.barh(A.index[::-1], A.test_f1[::-1], xerr=A.test_sd.fillna(0)[::-1], color=cols[::-1], capsize=3)
for i, v in enumerate(A.test_f1[::-1]): ax.text(v + .003, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlim(0.6, 0.75); ax.set_xlabel("Test macro-F1 (mean of 2 seeds)"); ax.grid(axis="x", alpha=.3)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig7_2_kd_ablations.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG); log = log[log.file != "fig7_2_kd_ablations"]
pd.concat([log, pd.DataFrame([dict(file="fig7_2_kd_ablations", step="7",
    caption="KD ablations on PlantWild P2 (MobileNetV3-L): teacher source, cross-fitting, calibration and label weight.")])]).to_csv(LOG, index=False)
print("saved fig7_2_kd_ablations\nStep 7.5 DONE.")

/tmp/ipykernel_58/342098021.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt.step(c); return float(lt.exp())


_dino      T_val=0.796  train acc=0.7772  mean p(given)=0.6240
_clip      T_val=1.058  train acc=0.6897  mean p(given)=0.5904
_insample  T_val=1.661  train acc=1.0000  mean p(given)=0.9903
trainer patched
pair 1/5 done (7.2 min)
pair 2/5 done (14.4 min)
pair 3/5 done (21.5 min)
pair 4/5 done (28.6 min)
pair 5/5 done (35.7 min)

=== KD ABLATIONS: PlantWild P2, MobileNetV3-L, seeds 0-1 ===
                                   n  val_f1  test_f1  test_sd  test_acc  vs_full_pp
variant                                                                            
Baseline (no KD)                  2  0.6599   0.6591   0.0028    0.7061       -5.14
DINOv2-only teacher               2  0.6862   0.7036   0.0017    0.7385       -0.69
CLIP-only teacher                 2  0.6712   0.6778   0.0026    0.7167       -3.27
In-sample teacher (no cross-fit)  2  0.6913   0.7049   0.0031    0.7384       -0.56
No calibration (T=1)              2  0.6711   0.6957   0.0026    0.7336       -1.48
alpha=1.0 (teacher o

In [16]:
# ===== Step 7.6: KD analysis from saved logits (expert subset, per-class, calibration) =====
import re, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import f1_score
W, FEAT, FIG, STU = "/kaggle/working", "/kaggle/working/feats", "/kaggle/working/figures", "/kaggle/working/students"
M = {t: pd.read_csv(f"{FEAT}/{t}_manifest.csv", low_memory=False) for t in ["pd", "pw"]}
NICE = {"mobilenetv3_large_100": "MobileNetV3-L", "efficientnet_b0": "EfficientNet-B0"}
def is_healthy(n):
    n = n.lower().replace("_", " ")
    return ("healthy" in n) or (bool(re.fullmatch(r"[a-z]+( [a-z]+)? leaf", n)) and not any(
        k in n for k in ["rust", "blight", "spot", "mildew", "scab", "rot", "virus", "mold", "mosaic"]))
def load(tag):
    Lg = np.load(f"{STU}/logits/{tag}_test.npy").astype(np.float32)
    return Lg, pd.read_csv(f"{STU}/logits/{tag}_test_uids.csv").row.values
def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1); e = 0.0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (conf > lo) & (conf <= lo + 1/bins)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return e
sm = lambda z: np.exp(z - z.max(1, keepdims=True)) / np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True)

# collect predictions: PW P2 seeds 0-2 (per run), PD P3 folds pooled (per model)
runs = []
for m in NICE:
    for s in [0, 1, 2]:
        for meth, suf in [("baseline", ""), ("kd", "_kd0.5t4")]:
            tag = f"pw_p2_{m}_s{s}_lr0.001{suf}"
            if os.path.exists(f"{STU}/logits/{tag}_test.npy"): runs.append(("pw", m, meth, f"seed{s}", *load(tag)))
    for meth, suf in [("baseline", ""), ("kd", "_kd0.5t4")]:
        parts = [load(f"pd_p3_f{k}_{m}_s0_lr0.001{suf}") for k in range(5)
                 if os.path.exists(f"{STU}/logits/pd_p3_f{k}_{m}_s0_lr0.001{suf}_test.npy")]
        if len(parts) == 5: runs.append(("pd", m, meth, "pooled5", np.vstack([p[0] for p in parts]), np.concatenate([p[1] for p in parts])))
print("runs loaded:", len(runs))

# 1) metrics incl. expert-verified subset + ECE
ev = M["pw"].expert_verified_test.astype(str).str.lower().isin(["true", "1", "1.0"]).values
rows, per_class = [], []
for d, m, meth, unit, Lg, idx in runs:
    y = M[d].class_id.values[idx]; p = sm(Lg); yp = p.argmax(1)
    row = dict(data=d, model=NICE[m], method=meth, unit=unit, n=len(y),
               f1_all=f1_score(y, yp, average="macro"), acc_all=(yp == y).mean(), ece=ece(p, y))
    if d == "pw":
        e = ev[idx]; row.update(n_expert=int(e.sum()), f1_expert=f1_score(y[e], yp[e], average="macro"), acc_expert=(yp[e] == y[e]).mean())
    rows.append(row)
    f1c = f1_score(y, yp, average=None, labels=np.arange(M[d].class_id.max() + 1), zero_division=0)
    for c, v in enumerate(f1c): per_class.append(dict(data=d, model=NICE[m], method=meth, unit=unit, class_id=c, f1=v))
T = pd.DataFrame(rows)
S = T.groupby(["data", "model", "method"]).agg(runs=("f1_all", "size"), f1_all=("f1_all", "mean"), f1_expert=("f1_expert", "mean"),
                                               acc_expert=("acc_expert", "mean"), ece=("ece", "mean")).round(4)
pd.set_option("display.width", 200); print("\n=== KD vs BASELINE: all test / expert-verified (PW) / calibration ===\n", S.to_string())
S.to_csv(f"{FIG}/table_kd_expert_calibration.csv")
if T.n_expert.notna().any(): print("expert-verified test images:", int(T.n_expert.dropna().iloc[0]))

# 2) per-class gains
PC = pd.DataFrame(per_class).groupby(["data", "class_id", "method"]).f1.mean().unstack("method").reset_index()
PC["gain_pp"] = 100 * (PC.kd - PC.baseline)
for d in ["pd", "pw"]:
    df = M[d]; names = df.drop_duplicates("class_id").set_index("class_id").class_name
    ntr = df[df[("p2_role")] .isin(["train"])].class_id.value_counts()
    sel = PC.data == d
    PC.loc[sel, "class_name"] = PC.loc[sel, "class_id"].map(names)
    PC.loc[sel, "n_train"] = PC.loc[sel, "class_id"].map(ntr).fillna(0).values
PC["healthy"] = PC.class_name.map(is_healthy)
PC["size_tercile"] = PC.groupby("data").n_train.transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["small", "medium", "large"]))
PC.to_csv(f"{FIG}/table_kd_per_class.csv", index=False)
G = PC.groupby(["data", "healthy"]).agg(classes=("gain_pp", "size"), mean_gain_pp=("gain_pp", "mean"),
                                        improved=("gain_pp", lambda g: f"{(g > 0).sum()}/{len(g)}")).round(2)
print("\n=== PER-CLASS GAIN: healthy vs disease ===\n", G.to_string())
G2 = PC.groupby(["data", "size_tercile"], observed=True).agg(classes=("gain_pp", "size"), mean_gain_pp=("gain_pp", "mean"),
                                                            base_f1=("baseline", "mean")).round(3)
print("\n=== PER-CLASS GAIN by training-set size ===\n", G2.to_string())
for d in ["pd", "pw"]:
    s = PC[PC.data == d].sort_values("gain_pp")
    print(f"\n{d}: biggest gains:", [(n[:25], round(g, 1)) for n, g in zip(s.class_name.tail(5), s.gain_pp.tail(5))])
    print(f"{d}: biggest losses:", [(n[:25], round(g, 1)) for n, g in zip(s.class_name.head(5), s.gain_pp.head(5))])

# 3) figures
LOG = f"{FIG}/figure_log.csv"; log = pd.read_csv(LOG)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 2.8]})
for ax, d, title in zip(axs, ["pd", "pw"], ["PlantDoc (P3 pooled)", "PlantWild (P2, 3 seeds)"]):
    s = PC[PC.data == d].sort_values("gain_pp")
    ax.bar(range(len(s)), s.gain_pp, color=np.where(s.healthy, "#55A868", "#C44E52"))
    ax.axhline(0, color="k", lw=.8); ax.set_title(title); ax.set_xticks([]); ax.set_xlabel("Classes (sorted by gain)")
axs[0].set_ylabel("KD − baseline per-class F1 (pp)")
axs[1].bar([], [], color="#C44E52", label="disease"); axs[1].bar([], [], color="#55A868", label="healthy"); axs[1].legend(fontsize=8)
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig7_3_per_class_gain.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
fig, axs = plt.subplots(1, 2, figsize=(8, 3.6), sharey=True)
for ax, d in zip(axs, ["pd", "pw"]):
    for meth, c in [("baseline", "#8C8C8C"), ("kd", "#4C72B0")]:
        rs = [r for r in runs if r[0] == d and r[2] == meth]
        p = np.vstack([sm(r[4]) for r in rs]); y = np.concatenate([M[d].class_id.values[r[5]] for r in rs])
        conf, ok = p.max(1), p.argmax(1) == y; bins = np.linspace(0, 1, 11); mid, acc = [], []
        for lo, hi in zip(bins[:-1], bins[1:]):
            mm = (conf > lo) & (conf <= hi)
            if mm.sum() > 20: mid.append(conf[mm].mean()); acc.append(ok[mm].mean())
        ax.plot(mid, acc, "o-", color=c, label=f"{meth} (ECE {ece(p, y):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_title({"pd": "PlantDoc", "pw": "PlantWild"}[d])
    ax.set_xlabel("Confidence"); ax.legend(fontsize=8)
axs[0].set_ylabel("Accuracy")
for e in ["png", "pdf"]: fig.savefig(f"{FIG}/fig7_4_calibration.{e}", dpi=300, bbox_inches="tight")
plt.close(fig)
log = log[~log.file.isin(["fig7_3_per_class_gain", "fig7_4_calibration"])]
pd.concat([log, pd.DataFrame([
    dict(file="fig7_3_per_class_gain", step="7", caption="Per-class F1 change from KD (MobileNetV3-L and EfficientNet-B0 averaged)."),
    dict(file="fig7_4_calibration", step="7", caption="Reliability diagrams of baseline vs KD students (pooled over models and runs).")])]).to_csv(LOG, index=False)
print("\nsaved fig7_3_per_class_gain, fig7_4_calibration\nStep 7.6 DONE.")

runs loaded: 16

=== KD vs BASELINE: all test / expert-verified (PW) / calibration ===
                                runs  f1_all  f1_expert  acc_expert     ece
data model           method                                               
pd   EfficientNet-B0 baseline     1  0.6254        NaN         NaN  0.1675
                     kd           1  0.6618        NaN         NaN  0.0493
     MobileNetV3-L   baseline     1  0.6142        NaN         NaN  0.1588
                     kd           1  0.6515        NaN         NaN  0.0463
pw   EfficientNet-B0 baseline     3  0.6638     0.4939      0.7498  0.1267
                     kd           3  0.7168     0.5470      0.7883  0.0425
     MobileNetV3-L   baseline     3  0.6561     0.4992      0.7480  0.1562
                     kd           3  0.7086     0.5473      0.7802  0.0464
expert-verified test images: 1151

=== PER-CLASS GAIN: healthy vs disease ===
               classes  mean_gain_pp improved
data healthy                          

In [17]:
   import os; os.system("cd /kaggle/working && rm -f wildleaf_step7.zip && zip -qr wildleaf_step7.zip students figures feats train_student.py -x 'students/ckpt/*'"); print(os.path.getsize("/kaggle/working/wildleaf_step7.zip")/1e6, "MB")

273.47862 MB
